# Processo de KDD: Classificação de Água (Limpa vs Suja)
Neste notebook, realizamos a etapa de mineração de dados (Data Mining) do processo de KDD. 
O objetivo é avaliar múltiplos algoritmos de classificação e selecionar o melhor para o nosso conjunto de dados de histogramas RGB (
es.csv).

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import normalize
import joblib

## 1. Carregamento e Pré-processamento
O dataset contém 768 colunas representando as frequências do histograma RGB da imagem.
Vamos carregar os dados e aplicar uma normalização (L1) para garantir que a soma do histograma seja 1 (independente do tamanho da imagem).

In [ ]:
# Carregando a base de dados
df = pd.read_csv('res.csv')

# Separando características (X) e a classe alvo (y)
X = df.drop('class', axis=1)
y = df['class']

# Normalização L1 dos histogramas
X_norm = normalize(X, norm='l1')
X = pd.DataFrame(X_norm, columns=X.columns)

print(f'Total de instâncias: {len(df)}')
print(df['class'].value_counts())

## 2. Metodologia de Avaliação
Vamos submeter o conjunto de dados a diversos algoritmos. Utilizaremos **Validação Cruzada (Cross-Validation)** com 5 folds para garantir uma avaliação robusta, dado o tamanho do nosso dataset.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

best_model_name = None
best_score = 0
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('--- Avaliando os Modelos ---')
for name, model in models.items():
    cv_results = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
    mean_score = cv_results.mean()
    print(f'{name}: Acurácia Média = {mean_score:.4f} (Desvio Padrão = {cv_results.std():.4f})')
    
    if mean_score > best_score:
        best_score = mean_score
        best_model_name = name

## 3. Seleção e Treinamento do Modelo Vencedor
Após avaliar os modelos, treinamos o melhor algoritmo com **toda a base de dados** e salvamos o modelo final para ser utilizado na aplicação Web.

In [ ]:
print(f'\nModelo Vencedor: {best_model_name} (Acurácia de {best_score:.4f})')

# Treinar com o algoritmo vencedor todo o conjunto de dados
best_model = models[best_model_name]
best_model.fit(X, y)

# Exportando o modelo para consumo do Front-end Web
joblib.dump(best_model, 'best_model.pkl')
print('O modelo final foi salvo como best_model.pkl com sucesso!')